# 05 - Gobernanza y documentación

**Objetivo.** Reunir la evidencia de metadatos de Unity Catalog para catálogo, esquemas, Volumes y tabla analítica. Este notebook no modifica datos.

**Validación.** Los comandos `DESCRIBE ... EXTENDED` muestran comentarios, propietario y ubicación. En Free Edition se reportan los permisos visibles para el usuario del workspace.

In [0]:
from pyspark.sql import functions as F

CATALOG = "oceanwatch_g06"
RAW_VOLUME = f"{CATALOG}.landing.raw_ais"
WPI_VOLUME = f"{CATALOG}.reference.world_port_index"
BENCHMARK_VOLUME = f"{CATALOG}.analytics.benchmark_files"
DELTA_TABLE = f"{CATALOG}.analytics.ais_positions_d07_delta"

## Completar y normalizar comentarios

In [0]:
# Los comentarios identifican propósito, procedencia y el carácter no productivo del benchmark.
spark.sql(f"COMMENT ON VOLUME {RAW_VOLUME} IS 'Archivos AIS NOAA crudos y CSV extraídos; fuente de ingesta, no tabla analítica.'")
spark.sql(f"COMMENT ON VOLUME {WPI_VOLUME} IS 'World Port Index NGA Pub. 150; referencia para asociar celdas H3 y puertos.'")
spark.sql(f"COMMENT ON VOLUME {BENCHMARK_VOLUME} IS 'Línea base Parquet del experimento de almacenamiento; evidencia reproducible, no artefacto productivo.'")
spark.sql(f"COMMENT ON TABLE {DELTA_TABLE} IS 'Tabla analítica oficial para historial temporal por MMSI; procede de AIS NOAA junio 2023 y aplica D-07: deduplicación exacta y MMSI válidos de nueve dígitos.'")

DataFrame[]

## Evidencia de catálogo, esquemas, Volumes y tabla

In [0]:
display(spark.sql(f"DESCRIBE CATALOG EXTENDED {CATALOG}"))
for schema_name in ("landing", "reference", "analytics"):
    display(spark.sql(f"DESCRIBE SCHEMA EXTENDED {CATALOG}.{schema_name}"))

for volume_name in (RAW_VOLUME, WPI_VOLUME, BENCHMARK_VOLUME):
    display(spark.sql(f"DESCRIBE VOLUME {volume_name}"))

display(spark.sql(f"DESCRIBE TABLE EXTENDED {DELTA_TABLE}"))

info_name,info_value
Catalog Name,oceanwatch_g06
Comment,Lakehouse OceanWatch Analytics para tráfico marítimo AIS de NOAA.
Owner,arangurenmarita@gmail.com
Catalog Type,Regular
Created By,arangurenmarita@gmail.com
Created At,2026-09-25 AD at 16:29:07 UTC
Updated By,arangurenmarita@gmail.com
Updated At,2026-09-25 AD at 16:29:07 UTC
Storage Root,
Storage Location,


database_description_item,database_description_value
Catalog Name,oceanwatch_g06
Namespace Name,landing
Comment,Datos AIS descargados y descomprimidos desde la fuente oficial NOAA.
Collation,UTF8_BINARY
Location,
Owner,arangurenmarita@gmail.com
Properties,"((unity.catalog.managed.iceberg.defaults.delta.feature.catalogManaged,supported))"
Predictive Optimization,ENABLE (inherited from METASTORE metastore_aws_us_east_2)


database_description_item,database_description_value
Catalog Name,oceanwatch_g06
Namespace Name,reference
Comment,Datos de referencia: World Port Index y catálogo de tipos AIS.
Collation,UTF8_BINARY
Location,
Owner,arangurenmarita@gmail.com
Properties,"((unity.catalog.managed.delta.defaults.delta.parquet.format.version,2.12.0), (unity.catalog.managed.delta.defaults.delta.parquet.format.version.afe.internal,2.12.0), (unity.catalog.managed.iceberg.defaults.delta.feature.catalogManaged,supported))"
Predictive Optimization,ENABLE (inherited from METASTORE metastore_aws_us_east_2)


database_description_item,database_description_value
Catalog Name,oceanwatch_g06
Namespace Name,analytics
Comment,Tablas optimizadas y resultados analíticos de OceanWatch.
Collation,UTF8_BINARY
Location,
Owner,arangurenmarita@gmail.com
Properties,"((unity.catalog.managed.iceberg.defaults.delta.feature.catalogManaged,supported))"
Predictive Optimization,ENABLE (inherited from METASTORE metastore_aws_us_east_2)


name,catalog,database,owner,storage_location,volume_type,comment,securable_type,securable_kind
raw_ais,oceanwatch_g06,landing,arangurenmarita@gmail.com,,MANAGED,"Archivos AIS NOAA crudos y CSV extraídos; fuente de ingesta, no tabla analítica.",VOLUME,VOLUME_DB_STORAGE


name,catalog,database,owner,storage_location,volume_type,comment,securable_type,securable_kind
world_port_index,oceanwatch_g06,reference,arangurenmarita@gmail.com,,MANAGED,World Port Index NGA Pub. 150; referencia para asociar celdas H3 y puertos.,VOLUME,VOLUME_DB_STORAGE


name,catalog,database,owner,storage_location,volume_type,comment,securable_type,securable_kind
benchmark_files,oceanwatch_g06,analytics,arangurenmarita@gmail.com,,MANAGED,"Línea base Parquet del experimento de almacenamiento; evidencia reproducible, no artefacto productivo.",VOLUME,VOLUME_DB_STORAGE


col_name,data_type,comment
MMSI,string,null
BaseDateTime,string,null
LAT,double,null
LON,double,null
SOG,double,null
COG,double,null
Heading,double,null
VesselName,string,null
IMO,string,null
CallSign,string,null


## Organización y permisos visibles

En Databricks Free Edition el workspace se opera bajo el usuario propietario. Se consulta la metadata efectiva, sin inventar `GRANT` adicionales.

In [0]:
display(spark.sql(f"SHOW GRANTS ON CATALOG {CATALOG}"))
display(spark.sql(f"SHOW GRANTS ON SCHEMA {CATALOG}.analytics"))
display(spark.sql(f"SHOW GRANTS ON TABLE {DELTA_TABLE}"))

Principal,ActionType,ObjectType,ObjectKey


Principal,ActionType,ObjectType,ObjectKey


Principal,ActionType,ObjectType,ObjectKey


## Resumen ejecutivo

El catálogo `oceanwatch_g06` organiza el proyecto AIS en tres esquemas funcionales. `landing.raw_ais` conserva la fuente y `reference.world_port_index` la referencia WPI. `analytics.ais_positions_d07_delta` es el artefacto analítico oficial y documenta procedencia y D-07; `analytics.benchmark_files` conserva únicamente la línea base Parquet del requisito 4. Propietarios y permisos se reportan desde Unity Catalog según la configuración efectiva de Free Edition.